# 02 — Feature Engineering with Leakage Controls
This notebook generates technical and microstructure features via `src.features.pipeline.run_feature_pipeline`, validates alignment and NaN behavior, and saves reusable feature artifacts.

It is entirely configuration-driven using `configs/features/*.yaml` merged through `resolve_config`.

In [1]:
from __future__ import annotations

from copy import deepcopy
from pathlib import Path
import random
import warnings

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from src.utils.config_loader import resolve_config, load_yaml
from src.utils.seed import set_global_seed
from src.data.loader import load_raw_pair
from src.data.preprocessing import preprocess_pair
from src.features.pipeline import run_feature_pipeline
from src.features.technical import add_technical_features
from src.features.microstructure import add_microstructure_features

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 160)

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'configs').exists():
            return candidate
    raise FileNotFoundError('Could not locate repository root.')

ROOT = find_repo_root(Path.cwd())
BASE_CONFIG = resolve_config(root=str(ROOT))
SEED = int(BASE_CONFIG.get('training', {}).get('random_seed', 42))
set_global_seed(SEED, deterministic_torch=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PAIR = BASE_CONFIG['data']['pairs'][0]
print(f'ROOT={ROOT}')
print(f'PAIR={PAIR}, SEED={SEED}')

2026-03-13 03:11:25 | src.utils.seed | INFO | Global seed set to 42 (deterministic_torch=True)


ROOT=c:\Users\Nabeel\Desktop\frl-trading-framework
PAIR=EURUSD, SEED=42


In [2]:
raw_df = load_raw_pair(PAIR, raw_dir=BASE_CONFIG['data']['raw_data_dir'])
train_raw, test_raw = preprocess_pair(raw_df, BASE_CONFIG)
train_feat, test_feat = run_feature_pipeline(train_raw.copy(), test_raw.copy(), BASE_CONFIG)

print(f'raw rows={len(raw_df):,} | train rows={len(train_feat):,} | test rows={len(test_feat):,}')
print(f'train columns={len(train_feat.columns)} | test columns={len(test_feat.columns)}')

raw rows=10,000 | train rows=7,951 | test rows=2,000
train columns=26 | test columns=26


## Feature selection is dynamic from configuration
The active feature groups and normalization policy come from `features.pipeline` plus `features.technical` and `features.microstructure` blocks in merged YAML configuration.

In [3]:
pipeline_cfg = BASE_CONFIG.get('features', {}).get('pipeline', {})
technical_cfg = BASE_CONFIG.get('features', {}).get('technical', {})
micro_cfg = BASE_CONFIG.get('features', {}).get('microstructure', {})

selected_groups = pipeline_cfg.get('selected_feature_groups', [])
retained_raw = pipeline_cfg.get('retained_raw_columns', [])
feature_cols = [c for c in train_feat.columns if c not in retained_raw and c != 'micro_session_label']

display(pd.DataFrame([
    {
        'selected_feature_groups': ', '.join(selected_groups),
        'normalization_enabled': pipeline_cfg.get('normalization', {}).get('enabled', False),
        'scaler_type': pipeline_cfg.get('normalization', {}).get('scaler_type', 'none'),
        'technical_indicators': ', '.join(technical_cfg.get('enabled_indicators', [])),
        'session_proxy_enabled': micro_cfg.get('session_proxy_enabled', False),
        'feature_count': len(feature_cols),
    }
]))

display(train_feat[feature_cols].head(3))

,selected_feature_groups,normalization_enabled,scaler_type,technical_indicators,session_proxy_enabled,feature_count
0,"technical, microstructure",True,standard,"sma, ema, rsi, macd, bollinger, rolling_volati...",True,19


,tech_boll_lower,tech_boll_mid,tech_boll_upper,tech_boll_width,tech_ema_10,tech_ema_20,tech_ema_50,tech_logret_1,tech_macd,tech_macd_hist,tech_macd_signal,tech_roll_vol_20,tech_rsi_14,tech_sma_10,tech_sma_20,tech_sma_50,micro_price_change_rate_5,micro_realized_vol_20,micro_spread_proxy
0,0.150296,0.136792,0.122827,-0.273974,0.115619,0.133736,0.168133,-0.297905,-0.672524,0.044157,-0.724454,-1.022962,-0.494343,0.101988,0.136792,0.176350,0.033243,-1.022962,0.123066
1,0.148935,0.131526,0.113653,-0.344342,0.111886,0.130165,0.165441,-0.410375,-0.679407,0.015669,-0.723013,-1.272641,-0.688808,0.102189,0.131526,0.173848,-0.244311,-1.272641,1.027697
2,0.152645,0.128015,0.102898,-0.475384,0.115487,0.130427,0.164301,1.202279,-0.594261,0.247215,-0.703857,-1.018088,0.011018,0.106759,0.128015,0.171392,0.249303,-1.018088,1.347112


## No-lookahead validation
To test leakage risk, we recompute feature values on truncated histories and compare the latest available feature row against the full-history computation at the same timestamp.

In [11]:

def compute_unscaled_features(df: pd.DataFrame, config: dict) -> pd.DataFrame:
    cfg = deepcopy(config)
    cfg['features']['pipeline']['normalization']['enabled'] = False
    out = df.copy()
    groups = cfg.get('features', {}).get('pipeline', {}).get('selected_feature_groups', [])
    if 'technical' in groups:
        out = add_technical_features(out, cfg)
    if 'microstructure' in groups:
        out = add_microstructure_features(out, cfg)
    out = out.dropna().reset_index(drop=True)
    return out

# Compute full unscaled features
unscaled_full = compute_unscaled_features(train_raw.copy(), BASE_CONFIG)

# Select feature columns
feature_cols_unscaled = [
    c for c in unscaled_full.columns
    if c.startswith('tech_') or c.startswith('micro_')
]

# Determine probe indices
probe_indices = np.linspace(120, max(121, len(train_raw) - 5), num=5, dtype=int)
probe_indices = sorted(set(int(i) for i in probe_indices if i < len(train_raw)))

# Initialize probe results
probe_rows = []

for idx in probe_indices:
    truncated = train_raw.iloc[: idx + 1].copy()
    unscaled_trunc = compute_unscaled_features(truncated, BASE_CONFIG)
    
    if len(unscaled_trunc) == 0:
        continue

    ts = unscaled_trunc['timestamp'].iloc[-1]
    full_match = unscaled_full.loc[unscaled_full['timestamp'] == ts]
    
    if full_match.empty:
        continue

    full_row = full_match.iloc[-1]
    trunc_row = unscaled_trunc.iloc[-1]

    # Safely convert features to numeric
    feature_data_full = pd.to_numeric(full_row[feature_cols_unscaled], errors='coerce').values
    feature_data_trunc = pd.to_numeric(trunc_row[feature_cols_unscaled], errors='coerce').values

    # Compute max absolute difference
    max_abs_diff = float(np.nanmax(np.abs(feature_data_full - feature_data_trunc)))

    probe_rows.append({'timestamp': ts, 'max_abs_feature_diff': max_abs_diff})
    
    assert max_abs_diff < 1e-9, f'Potential lookahead discrepancy at {ts}: {max_abs_diff}'

# Create results DataFrame
leakage_probe_df = pd.DataFrame(probe_rows)
display(leakage_probe_df if not leakage_probe_df.empty else pd.DataFrame([{'status': 'no valid probes'}]))

,timestamp,max_abs_feature_diff
0,2010-03-02 11:00:00+00:00,0.0
1,2010-06-24 10:00:00+00:00,0.0
2,2010-10-18 11:00:00+00:00,0.0
3,2011-02-09 13:00:00+00:00,0.0
4,2011-06-03 13:00:00+00:00,0.0


## Output validation checks
We enforce schema alignment, timestamp ordering, and NaN-free engineered columns before writing reusable artifacts.

In [10]:
assert list(train_feat.columns) == list(test_feat.columns), 'Train/test feature schema mismatch'
assert train_feat['timestamp'].is_monotonic_increasing, 'Train timestamps are not sorted'
assert test_feat['timestamp'].is_monotonic_increasing, 'Test timestamps are not sorted'
assert train_feat['timestamp'].max() < test_feat['timestamp'].min(), 'Train/test temporal overlap detected'

nan_train = int(train_feat[feature_cols].isna().sum().sum())
nan_test = int(test_feat[feature_cols].isna().sum().sum())
assert nan_train == 0, f'Residual train NaNs in feature columns: {nan_train}'
assert nan_test == 0, f'Residual test NaNs in feature columns: {nan_test}'

alignment_df = pd.DataFrame([
    {
        'train_rows': len(train_feat),
        'test_rows': len(test_feat),
        'feature_columns': len(feature_cols),
        'nan_train': nan_train,
        'nan_test': nan_test,
        'first_train_ts': train_feat['timestamp'].iloc[0],
        'last_train_ts': train_feat['timestamp'].iloc[-1],
        'first_test_ts': test_feat['timestamp'].iloc[0],
        'last_test_ts': test_feat['timestamp'].iloc[-1],
    }
])
display(alignment_df)

,train_rows,test_rows,feature_columns,nan_train,nan_test,first_train_ts,last_train_ts,first_test_ts,last_test_ts
0,7951,2000,19,0,0,2010-02-25 12:00:00+00:00,2011-06-03 17:00:00+00:00,2011-06-03 18:00:00+00:00,2011-09-29 01:00:00+00:00


## Save feature artifacts for downstream training
Artifacts are saved to the canonical processed-data directory configured in `data.processed_data_dir`, enabling direct reuse in training and evaluation workflows.

In [ ]:
processed_root = Path(BASE_CONFIG['data']['processed_data_dir']) / PAIR
processed_root.mkdir(parents=True, exist_ok=True)

train_out = processed_root / 'train.parquet'
test_out = processed_root / 'test.parquet'
train_feat.to_parquet(train_out, index=False)
test_feat.to_parquet(test_out, index=False)

print(f'✅ Saved train features -> {train_out}')
print(f'✅ Saved test features  -> {test_out}')